# Notebook 44 — Exact Exponential Decay in the Final Collapsed Coordinate

This notebook follows Notebook 43.

Notebook 43 showed that when the response is expressed in the final collapsed coordinate,
the fitted effective-rate exponent is approximately zero:

\[
\Gamma_{\mathrm{eff}}(x) \propto x^p, \qquad p \approx 0.
\]

That means the dynamics are no longer well described as having a scale-dependent rate in the
**final** coordinate. Instead, they become approximately constant-rate:

\[
\frac{dS}{dx} = -\Gamma S.
\]

So the natural prediction is:

\[
S(x) \approx e^{-\Gamma x}.
\]

Main goals:
1. rebuild the final collapsed coordinate,
2. test whether \(\log S\) is linear in \(x\),
3. compare pure exponential vs stretched-exponential fits,
4. measure protocol-by-protocol deviations from exact exponential decay,
5. confirm whether the final coordinate is the one in which the noisy dynamics become simplest.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar, curve_fit
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Baseline protocol and nearby family

In [ ]:
baseline = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

T0 = baseline['T']
alpha0 = baseline['alpha']
omega0 = baseline['omega_max']

protocols = {
    'baseline': dict(baseline),
    'T_minus': {**baseline, 'T': 24.0},
    'T_plus': {**baseline, 'T': 30.0},
    'alpha_minus': {**baseline, 'alpha': 0.08},
    'alpha_plus': {**baseline, 'alpha': 0.12},
    'omega_minus': {**baseline, 'omega_max': 1.02},
    'omega_plus': {**baseline, 'omega_max': 1.16},
}


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    H = build_time_dependent_hamiltonian(
        params['T'], params['omega_max'], params['alpha'], V, params['delta0'], params['p'], params['q']
    )
    times = np.linspace(0.0, params['T'], n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            vals.append(basis_state.overlap(state))
        else:
            val = basis_state.dag() * state * basis_state
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            score = np.abs(basis_state.overlap(final_state)) ** 2
        else:
            val = basis_state.dag() * final_state * basis_state
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Noise grid

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 13)
gamma_phi_vals = np.linspace(0.0, 0.12, 13)


## Evaluate one protocol

In [ ]:
def evaluate_protocol(params):
    cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            U_eff, finals = build_noisy_effective_map(
                params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=140
            )
            cz_map[i, j] = compensated_cz_fidelity(U_eff)
            coh_map[i, j] = coherence_proxy(finals)
            leak_map[i, j] = leakage_from_finals(finals)

    S = cz_map * coh_map * (1.0 - leak_map)
    S_norm = S / np.max(S) if np.max(S) > 0 else S.copy()

    S_gamma = S_norm[0, :]
    S_phi = S_norm[:, 0]
    interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

    def loss(lam):
        mapped = lam * gamma_phi_vals
        pred = interp_gamma(np.clip(mapped, gamma_decay_vals.min(), gamma_decay_vals.max()))
        return float(np.mean((pred - S_phi) ** 2))

    fit = minimize_scalar(loss, bounds=(0.1, 5.0), method='bounded')
    lam = float(fit.x)

    gamma_eff_grid = np.zeros_like(S_norm)
    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            gamma_eff_grid[i, j] = gamma_decay + lam * gamma_phi

    return {
        'params': params,
        'S_norm': S_norm,
        'lambda': lam,
        'gamma_eff_grid': gamma_eff_grid,
    }


## Boundary laws for T_c

In [ ]:
def build_boundary_laws():
    baseline_res = evaluate_protocol(baseline)
    lambda0 = baseline_res['lambda']

    def score_for_params(params):
        res = evaluate_protocol(params)
        S = res['S_norm']
        S_gamma = S[0, :]
        S_phi = S[:, 0]
        interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

        def axis_loss(lam):
            mapped = lam * gamma_phi_vals
            pred = interp_gamma(np.clip(mapped, gamma_decay_vals.min(), gamma_decay_vals.max()))
            return float(np.mean((pred - S_phi) ** 2))

        aloss = axis_loss(res['lambda'])

        bS = baseline_res['S_norm']
        bge = baseline_res['gamma_eff_grid'].ravel()
        bsv = bS.ravel()
        order = np.argsort(bge)
        bge = bge[order]
        bsv = bsv[order]
        bbins = np.linspace(bge.min(), bge.max(), 13)
        bcenters = 0.5 * (bbins[:-1] + bbins[1:])
        bmeans = np.full(len(bcenters), np.nan)
        for k in range(len(bcenters)):
            mask = (bge >= bbins[k]) & (bge < bbins[k+1])
            if np.any(mask):
                bmeans[k] = np.mean(bsv[mask])
        bvalid = ~np.isnan(bmeans)
        interp0 = interp1d(bcenters[bvalid], bmeans[bvalid], kind='linear', fill_value='extrapolate')

        ge = res['gamma_eff_grid'].ravel()
        sv = S.ravel()
        order = np.argsort(ge)
        ge = ge[order]
        sv = sv[order]
        bins = np.linspace(ge.min(), ge.max(), 13)
        centers = 0.5 * (bins[:-1] + bins[1:])
        means = np.full(len(centers), np.nan)
        for k in range(len(centers)):
            mask = (ge >= bins[k]) & (ge < bins[k+1])
            if np.any(mask):
                means[k] = np.mean(sv[mask])
        valid = ~np.isnan(means)
        curve_mismatch = float(np.mean((means[valid] - interp0(np.clip(centers[valid], bcenters[bvalid].min(), bcenters[bvalid].max()))) ** 2))
        lambda_shift = abs(res['lambda'] - lambda0)

        return float(np.exp(
            -(lambda_shift / 0.15)
            -(aloss / 0.001)
            -(curve_mismatch / 0.001)
        ))

    boundary_level = 0.30
    T_vals_local = np.array([20.0, 24.0, 26.0, 30.0, 34.0])
    alpha_vals_local = np.array([0.06, 0.08, 0.10, 0.12, 0.14])
    omega_vals_local = np.array([0.95, 1.02, 1.09, 1.16, 1.23])

    TA_score = np.zeros((len(alpha_vals_local), len(T_vals_local)))
    TO_score = np.zeros((len(omega_vals_local), len(T_vals_local)))

    for i, alpha in enumerate(alpha_vals_local):
        for j, T in enumerate(T_vals_local):
            TA_score[i, j] = score_for_params({**baseline, 'T': float(T), 'alpha': float(alpha)})

    for i, omega in enumerate(omega_vals_local):
        for j, T in enumerate(T_vals_local):
            TO_score[i, j] = score_for_params({**baseline, 'T': float(T), 'omega_max': float(omega)})

    def extract_vertical_boundary(score_map, xvals, yvals, level):
        bx, by = [], []
        for i, y in enumerate(yvals):
            row = score_map[i, :]
            crossing = None
            for j in range(len(xvals) - 1):
                v0, v1 = row[j], row[j + 1]
                if (v0 >= level and v1 < level) or (v0 > level and v1 <= level):
                    x0, x1 = xvals[j], xvals[j + 1]
                    xc = x0 if v1 == v0 else x0 + (level - v0) * (x1 - x0) / (v1 - v0)
                    crossing = xc
                    break
            if crossing is None and np.max(row) >= level and np.min(row) >= level:
                crossing = float(xvals[-1])
            if crossing is not None:
                bx.append(float(crossing))
                by.append(float(y))
        return np.array(bx), np.array(by)

    TA_bx, TA_by = extract_vertical_boundary(TA_score, T_vals_local, alpha_vals_local, boundary_level)
    TO_bx, TO_by = extract_vertical_boundary(TO_score, T_vals_local, omega_vals_local, boundary_level)

    TA_x = TA_by / alpha0
    TA_y = TA_bx / T0
    TO_x = TO_by / omega0
    TO_y = TO_bx / T0

    coeff_alpha = np.polyfit(TA_x - 1.0, TA_y, deg=1)
    coeff_omega = np.polyfit(TO_x, TO_y, deg=1)
    return coeff_alpha, coeff_omega

coeff_alpha, coeff_omega = build_boundary_laws()
coeff_alpha, coeff_omega


## Define T_c, recover nu, and build the final collapsed dataset

In [ ]:
def Tc_over_T0_alpha(alpha):
    x = alpha / alpha0
    return np.polyval(coeff_alpha, x - 1.0)

def Tc_over_T0_omega(omega):
    x = omega / omega0
    return np.polyval(coeff_omega, x)

def Tc_from_params(params):
    Tc_alpha = T0 * Tc_over_T0_alpha(params['alpha'])
    Tc_omega = T0 * Tc_over_T0_omega(params['omega_max'])
    return float(np.sqrt(np.maximum(Tc_alpha, 1e-9) * np.maximum(Tc_omega, 1e-9)))

family = {}
for name, pset in protocols.items():
    family[name] = evaluate_protocol(pset)

def collapse_error_nu(nu):
    x_all = []
    s_all = []
    for name, res in family.items():
        T = res['params']['T']
        Tc = Tc_from_params(res['params'])
        x = res['gamma_eff_grid'].ravel() * (T / Tc) ** nu
        s = res['S_norm'].ravel()
        x_all.extend(list(x))
        s_all.extend(list(s))
    x_all = np.array(x_all, dtype=float)
    s_all = np.array(s_all, dtype=float)

    order = np.argsort(x_all)
    x_all = x_all[order]
    s_all = s_all[order]

    bins = np.linspace(x_all.min(), x_all.max(), 24)
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = np.full(len(centers), np.nan)
    for k in range(len(centers)):
        mask = (x_all >= bins[k]) & (x_all < bins[k+1])
        if np.any(mask):
            means[k] = np.mean(s_all[mask])

    valid = ~np.isnan(means)
    pred = interp1d(centers[valid], means[valid], kind='linear', fill_value='extrapolate')
    p = pred(np.clip(x_all, centers[valid].min(), centers[valid].max()))
    return float(np.mean((s_all - p) ** 2))

opt = minimize_scalar(collapse_error_nu, bounds=(0.1, 3.0), method='bounded')
nu_opt = float(opt.x)
print("Recovered nu =", nu_opt)

x_all = []
s_all = []
labels = []

for name, res in family.items():
    T = res['params']['T']
    Tc = Tc_from_params(res['params'])
    x = res['gamma_eff_grid'].ravel() * (T / Tc) ** nu_opt
    s = res['S_norm'].ravel()

    x_all.extend(list(x))
    s_all.extend(list(s))
    labels.extend([name] * len(x))

x_all = np.array(x_all, dtype=float)
s_all = np.array(s_all, dtype=float)
labels = np.array(labels, dtype=object)

order = np.argsort(x_all)
x_sorted = x_all[order]
s_sorted = s_all[order]

bins = np.linspace(x_sorted.min(), x_sorted.max(), 40)
centers = 0.5 * (bins[:-1] + bins[1:])
means = np.full(len(centers), np.nan)

for k in range(len(centers)):
    mask = (x_sorted >= bins[k]) & (x_sorted < bins[k+1])
    if np.any(mask):
        means[k] = np.mean(s_sorted[mask])

valid = ~np.isnan(means)
x_curve = centers[valid]
S_curve = means[valid]

print("Collapsed curve points:", len(x_curve))


## Fit pure exponential and stretched exponential

In [ ]:
def exp_law(x, a):
    return np.exp(-a * x)

def stretched_law(x, a, b):
    return np.exp(-a * np.power(x, b))

popt_exp, _ = curve_fit(
    exp_law,
    np.clip(x_curve, 1e-9, None),
    np.clip(S_curve, 1e-9, 1.0),
    p0=[20.0],
    bounds=([0.0], [500.0]),
    maxfev=50000
)
a_exp = float(popt_exp[0])
S_exp = exp_law(np.clip(x_curve, 1e-9, None), a_exp)

popt_st, _ = curve_fit(
    stretched_law,
    np.clip(x_curve, 1e-9, None),
    np.clip(S_curve, 1e-9, 1.0),
    p0=[5.0, 1.0],
    bounds=([0.0, 0.1], [100.0, 5.0]),
    maxfev=50000
)
a_st, b_st = [float(v) for v in popt_st]
S_st = stretched_law(np.clip(x_curve, 1e-9, None), a_st, b_st)

mse_exp = float(np.mean((S_curve - S_exp) ** 2))
mse_st = float(np.mean((S_curve - S_st) ** 2))

print("Pure exponential fit: a =", a_exp, "MSE =", mse_exp)
print("Stretched fit: a =", a_st, "b =", b_st, "MSE =", mse_st)


## Log-linear test: does log(S) become linear in x?

In [ ]:
mask_log = (S_curve > 0) & (S_curve < 1.0)
x_log = x_curve[mask_log]
y_log = np.log(np.clip(S_curve[mask_log], 1e-12, 1.0))

coeff_lin = np.polyfit(x_log, y_log, deg=1)
y_log_fit = coeff_lin[0] * x_log + coeff_lin[1]
r2_log = 1.0 - np.sum((y_log - y_log_fit) ** 2) / np.sum((y_log - np.mean(y_log)) ** 2)

print("log-linear slope =", coeff_lin[0])
print("log-linear intercept =", coeff_lin[1])
print("log-linear R^2 =", r2_log)


## Plot collapsed curve with exponential and stretched fits

In [ ]:
plt.figure(figsize=(8.2, 5.4))
plt.scatter(x_all, s_all, s=7, alpha=0.18, label='collapsed data')
plt.plot(x_curve, S_curve, linewidth=2.2, label='binned collapsed curve')
plt.plot(x_curve, S_exp, linewidth=2.2, label=f'pure exponential, a={a_exp:.4f}')
plt.plot(x_curve, S_st, linewidth=2.2, label=f'stretched fit, b={b_st:.4f}')
plt.xlabel('x = gamma_eff * (T/Tc)^nu')
plt.ylabel('S')
plt.title('Final collapsed law: exponential vs stretched')
plt.legend()
plt.tight_layout()
plt.show()


## Plot log(S) vs x

In [ ]:
plt.figure(figsize=(8.0, 5.0))
plt.scatter(x_log, y_log, s=12, alpha=0.55, label='collapsed curve')
plt.plot(x_log, y_log_fit, linewidth=2.3, label=f'linear fit, R^2={r2_log:.4f}')
plt.xlabel('x')
plt.ylabel('log S')
plt.title('Log-linear test in the final collapsed coordinate')
plt.legend()
plt.tight_layout()
plt.show()


## Protocol-by-protocol exponential residuals

In [ ]:
per_protocol = {}

for name, res in family.items():
    T = res['params']['T']
    Tc = Tc_from_params(res['params'])
    x = res['gamma_eff_grid'].ravel() * (T / Tc) ** nu_opt
    s = res['S_norm'].ravel()

    pred = exp_law(np.clip(x, 1e-9, None), a_exp)
    per_protocol[name] = {
        'mse': float(np.mean((s - pred) ** 2)),
        'mae': float(np.mean(np.abs(s - pred))),
    }

for name, stats in per_protocol.items():
    print(name, stats)


## Plot per-protocol deviations from the pure exponential law

In [ ]:
plt.figure(figsize=(8.2, 5.1))

for name, res in family.items():
    T = res['params']['T']
    Tc = Tc_from_params(res['params'])
    x = res['gamma_eff_grid'].ravel() * (T / Tc) ** nu_opt
    s = res['S_norm'].ravel()

    order = np.argsort(x)
    x = x[order]
    s = s[order]
    pred = exp_law(np.clip(x, 1e-9, None), a_exp)

    bins = np.linspace(x.min(), x.max(), 15)
    centers = 0.5 * (bins[:-1] + bins[1:])
    resid = np.full(len(centers), np.nan)
    for k in range(len(centers)):
        mask = (x >= bins[k]) & (x < bins[k+1])
        if np.any(mask):
            resid[k] = np.mean(s[mask] - pred[mask])
    valid = ~np.isnan(resid)
    plt.plot(centers[valid], resid[valid], label=name)

plt.axhline(0.0, linestyle='--')
plt.xlabel('x')
plt.ylabel('mean residual to pure exponential')
plt.title('Protocol-wise deviation from exact exponential decay')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Exact exponential decay in the final collapsed coordinate")
lines.append("")
lines.append(f"Recovered collapse exponent nu = {nu_opt:.6f}")
lines.append("")
lines.append(f"Pure exponential fit: S(x) ≈ exp(-{a_exp:.6f} x)")
lines.append(f"Pure exponential MSE = {mse_exp:.6e}")
lines.append("")
lines.append(f"Stretched fit: S(x) ≈ exp(-{a_st:.6f} x^{b_st:.6f})")
lines.append(f"Stretched-fit MSE = {mse_st:.6e}")
lines.append("")
lines.append(f"log-linear test: slope = {coeff_lin[0]:.6f}, intercept = {coeff_lin[1]:.6f}, R^2 = {r2_log:.6f}")
lines.append("")
lines.append("Protocol-wise residuals to the pure exponential law:")
for name, stats in per_protocol.items():
    lines.append(
        f"- {name}: MSE={stats['mse']:.6e}, MAE={stats['mae']:.6e}"
    )
lines.append("")
lines.append("Interpretation:")
lines.append("- If log(S) is close to linear in x, then the final collapsed coordinate is the one in which the noisy dynamics become simplest.")
lines.append("- A strong log-linear fit means the stretched behavior seen earlier is mostly removed by the correct coordinate choice.")
lines.append("- Residual protocol structure reveals how close the collapse is to exact exponential universality rather than approximate universality.")

summary_text = "\n".join(lines)
print(summary_text)


## Final conclusion

This notebook tests the strongest possible statement in the project:

> in the final collapsed coordinate, does the response become exactly exponential?

The answer is checked directly by fitting

\[
S(x) \approx e^{-\Gamma x}
\]

and by testing whether

\[
\log S
\]

is linear in \(x\).

If that linearity holds well, then the final collapsed coordinate is the one in which the open-system Rydberg dynamics become simplest: an approximately constant-rate decay law.

That is the natural continuation after Notebook 43 because it turns the fitted rate constancy into the direct observable test that matters most.
